# Tokenization - Aarabhi Kuchi

In [9]:
import json, re
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load one row per contract

Each contract repeats 41 times, so we want to get contracts with no duplicates.

In [10]:
with open('../data/train_separate_questions.json', "r") as file:
    train_json = json.load(file)
with open('../data/test.json', "r") as file:
    test_json = json.load(file)

In [12]:

def get_contracts(cuad_json):
    contracts = []
    for contract in cuad_json['data']:
        for paragraph in contract['paragraphs']:
            contracts.append({"title": contract['title'], "context": paragraph['context']})
    return pd.DataFrame(contracts)


In [13]:
train_contracts = get_contracts(train_json)

In [14]:
test_contracts = get_contracts(test_json)

In [ ]:
train_contracts.shape

(408, 2)


In [16]:
train_contracts['title'].is_unique

True

In [17]:
train_contracts.head(5)

,title,context
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,EXHIBIT 10.6\n\n ...
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...
2,NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...
3,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...
4,"KIROMICBIOPHARMA,INC_05_11_2020-EX-10.23-CONSU...",Exhibit 10.23 Corporate Address Fannin South P...


In [18]:
test_contracts.shape

(102, 2)

In [19]:
test_contracts['title'].is_unique

True

In [21]:
train_contracts['context'].str.len().describe()

count       408.000000
mean      53991.710784
std       55985.180326
min        1081.000000
25%       16569.750000
50%       35960.500000
75%       68439.750000
max      338211.000000
Name: context, dtype: float64

Contracts range from about 1081 to 338211 characters, with a median of 35960. The mean is above the median, so most contracts are moderate in length and a few very long ones pull the average up. A transformer can only read about 512 tokens (~2000 characters) at once, so a typical contract is 15-20x too long. So chunking is needed.

## 2. Tokenizer

Function tokenize_words(text) that returns a list of lowercase words in the contracts.

All lowercase text, numbers included as numbers are important in contracts. Will figure out stopwords based on frequency of words and if they are relevant. Including hyphens as details are important in contracts.

In [23]:
def tokenize_words(text):
    lower_text = text.lower()
    pattern = r'[a-z0-9]+(?:[-\'][a-z0-9]+)*'
    return re.findall(pattern, lower_text)

In [24]:
tokenize_words("The Licensee shall NOT assign this Non-Compete Agreement within 30 days.")

['the',
 'licensee',
 'shall',
 'not',
 'assign',
 'this',
 'non-compete',
 'agreement',
 'within',
 '30',
 'days']

## 3. Test on Real Clauses

Checking if the tokenizer works on real legal text

In [27]:
def get_clauses(cuad_json, category):
    clauses = []
    for contract in cuad_json['data']:
        for paragraph in contract['paragraphs']:
            if paragraph['category'] == category:
                clauses.append({"title": contract['title'], "context": paragraph['context']})
    return pd.DataFrame(clauses)


In [29]:
len(get_clauses(train_json, "Confidentiality"))

KeyError: 'category'

## 4. Contract Length

## 5. Summary